# Bab 25: Text Mining Dasar

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab25/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab25")
print("Folder kerja:", os.getcwd())

## Listing 25.1: Fungsi prapemrosesan teks

`kode/bab25/normalisasi.py`

In [ ]:
import re
from functools import lru_cache
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import (
    StopWordRemoverFactory)

KAMUS = {"bgt": "banget", "ga": "tidak", "gak": "tidak",
         "nggak": "tidak", "tp": "tapi", "udah": "sudah",
         "aja": "saja", "hp": "ponsel", "app": "aplikasi",
         "appnya": "aplikasinya", "mantul": "mantap",
         "asem": "asam", "kayak": "seperti", "buat": "untuk",
         "nugas": "mengerjakan_tugas", "lemot": "lambat"}
GABUNG = {"force close": "force_close", "worth it": "worth_it",
          "v60": "v_enam_puluh", "ac nya": "acnya"}
NEGASI = {"tidak", "tak", "bukan", "belum", "kurang", "jangan"}
JAGA = NEGASI | {"tapi", "masih", "lama", "sudah", "terlalu",
                 "sangat", "banget", "naik", "beda", "bisa"}
STOP = set(StopWordRemoverFactory().get_stop_words()) - JAGA
_stem = StemmerFactory().create_stemmer()


@lru_cache(maxsize=None)
def stem(kata):
    return _stem.stem(kata) if "_" not in kata else kata


def bersihkan(teks, stopword=True, stemming=True, negasi=True):
    t = teks.lower()
    for a, b in GABUNG.items():
        t = t.replace(a, b)
    token = re.findall(r"[a-z_]+", t)
    token = [KAMUS.get(w, w) for w in token]
    if stopword:
        token = [w for w in token if w not in STOP]
    if stemming:
        token = [stem(w) for w in token]
    if negasi:                       # tidak enak -> tidak_enak
        hasil, i = [], 0
        while i < len(token):
            if token[i] in NEGASI and i + 1 < len(token):
                hasil.append(token[i] + "_" + token[i + 1]); i += 2
            else:
                hasil.append(token[i]); i += 1
        token = hasil
    return " ".join(token)

## Listing 25.2: Prapemrosesan langkah demi langkah

`kode/bab25/prapemrosesan.py`

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import (
    StopWordRemoverFactory)
from normalisasi import KAMUS, STOP, bersihkan, stem

t = "Kopinya ENAK bgt, tp antrinya lama dan AC nya ga dingin di sini"
print("asli  :", t)
print("norm  :", bersihkan(t, False, False, False))
print("+stop :", bersihkan(t, True, False, False))
print("+stem :", bersihkan(t, True, True, False))
print("+neg  :", bersihkan(t))

for w in ["pelayanannya", "pengirimannya", "kemanisan",
          "kebanyakan", "diperbaiki", "baristanya", "menyenangkan",
          "berkualitas", "pegawainya", "kemahalan"]:
    print(f"  {w:>13s} -> {stem(w)}")

asli = set(StopWordRemoverFactory().get_stop_words())
print(f"\nstopword Sastrawi: {len(asli)} kata")
print("dikeluarkan dari daftar:", ", ".join(sorted(asli - STOP)))
print("contoh kalimat negatif dengan stopword bawaan:")
print("  ", " ".join(w for w in "kopinya tidak enak tapi murah"
                     .split() if w not in asli))

## Listing 25.3: Bag of words, TF-IDF, dan kosinus

`kode/bab25/tfidf_mini.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import (CountVectorizer,
                                             TfidfVectorizer)

dok = ["kopi enak kopi wangi",
       "kopi pahit pelayanan lama",
       "pelayanan ramah tempat nyaman"]
cv = CountVectorizer()
X = cv.fit_transform(dok).toarray()
kata = cv.get_feature_names_out()
print(pd.DataFrame(X, columns=kata).to_string())

n = len(dok)
df = (X > 0).sum(0)
idf = np.log((1 + n) / (1 + df)) + 1
for v in (1, 2):
    print(f"df = {v}: idf = {np.log((1 + n) / (1 + v)) + 1:.3f}  ",
          ", ".join(kata[df == v]))
W = X * idf
W = W / np.linalg.norm(W, axis=1, keepdims=True)
print("TF-IDF dokumen 1:", {k: round(float(v), 3) for k, v
                            in zip(kata, W[0]) if v > 0})

tv = TfidfVectorizer()
W2 = tv.fit_transform(dok).toarray()
print("sama dengan sklearn:", np.allclose(W, W2))
cos = W2 @ W2.T
print("kemiripan kosinus:\n", cos.round(3))

## Listing 25.4: Klasifikasi sentimen ulasan

`kode/bab25/sentimen.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.naive_bayes import MultinomialNB
from normalisasi import bersihkan

d = pd.read_csv("kode/data/ulasan.csv", parse_dates=["tanggal"])
latih = d.tanggal < "2026-02-01"
print(f"latih {latih.sum()} ulasan, uji {(~latih).sum()} ulasan")
print("sebaran sentimen:", d.sentimen.value_counts().to_dict())

varian = {"mentah": dict(stopword=False, stemming=False,
                         negasi=False),
          "stop+stem": dict(negasi=False),
          "stop+stem+negasi": {}}
print("\nmakro-F1 data uji (unigram+bigram TF-IDF):")
print(f"{'praproses':18s} {'NB':>6s} {'logistik':>9s}")
for nama, opsi in varian.items():
    teks = d.teks.map(lambda t: bersihkan(t, **opsi))
    tv = TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                         sublinear_tf=True)
    Xl, Xu = tv.fit_transform(teks[latih]), tv.transform(teks[~latih])
    f = []
    for m in (MultinomialNB(alpha=0.5),
              LogisticRegression(C=5, max_iter=2000,
                                 class_weight="balanced")):
        m.fit(Xl, d.sentimen[latih])
        f.append(f1_score(d.sentimen[~latih], m.predict(Xu),
                          average="macro"))
    print(f"{nama:18s} {f[0]:6.3f} {f[1]:9.3f}   "
          f"({Xl.shape[1]} atribut)")

pred = m.predict(Xu)
print("\nlogistik, praproses lengkap:")
print(classification_report(d.sentimen[~latih], pred, digits=3))

kata = tv.get_feature_names_out()
for kelas in ("negatif", "positif"):
    k = list(m.classes_).index(kelas)
    top = np.argsort(m.coef_[k])[::-1][:6]
    print(f"{kelas}: " + ", ".join(kata[top]))

uji = d[~latih].assign(tebakan=pred)
salah = uji[uji.tebakan != uji.sentimen]
print(f"\n{len(salah)} salah tebak, contoh:")
for _, r in salah.sample(4, random_state=3).iterrows():
    print(f"  [{r.rating}] {r.teks[:60]}")
    print(f"      label {r.sentimen}, tebakan {r.tebakan}")

## Listing 25.5: LDA dan NMF untuk ulasan

`kode/bab25/topik_lda.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import (CountVectorizer,
                                             TfidfVectorizer)
from normalisasi import bersihkan

d = pd.read_csv("kode/data/ulasan.csv")
teks = d.teks.map(bersihkan)
buang = ["jujur", "overall", "coba", "kali", "top", "recommended",
         "sukses", "terus", "balik", "pasti", "sejauh", "parah",
         "kecewa", "lumayan", "biasa", "tolong", "deh", "sih"]
label = d.topik.str.split(";")


def tampilkan(nama, model, kata, W):
    print(f"\n{nama}:")
    utama = W.argmax(1)
    for j, komp in enumerate(model.components_):
        top = ", ".join(kata[np.argsort(komp)[::-1][:6]])
        grup = label[utama == j]
        lab = grup.explode().value_counts() / len(grup)
        print(f"  {j}: {top}")
        print(f"     {len(grup)} ulasan; {lab.index[0]} di "
              f"{lab.iloc[0]:.0%}, {lab.index[1]} di "
              f"{lab.iloc[1]:.0%}")


cv = CountVectorizer(stop_words=buang, min_df=5)
Xc = cv.fit_transform(teks)
lda = LatentDirichletAllocation(6, learning_method="batch",
                                random_state=0, max_iter=50)
tampilkan("LDA", lda, cv.get_feature_names_out(),
          lda.fit_transform(Xc))

tv = TfidfVectorizer(stop_words=buang, min_df=5)
Xt = tv.fit_transform(teks)
nmf = NMF(6, init="nndsvda", random_state=0, max_iter=500)
tampilkan("NMF", nmf, tv.get_feature_names_out(),
          nmf.fit_transform(Xt))

## Listing 25.6: Pemilahan ulasan dan dasbor keluhan

`kode/bab25/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from normalisasi import bersihkan

d = pd.read_csv("kode/data/ulasan.csv", parse_dates=["tanggal"])
d["bersih"] = d.teks.map(bersihkan)
latih = (d.tanggal < "2026-02-01").to_numpy()
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(d.topik.str.split(";"))
TOPIK = list(mlb.classes_)

KUNCI = {"antar": ["kirim", "tumpah", "cair", "kemas"],
         "aplikasi": ["aplikasi", "login", "error", "poin"],
         "harga": ["harga", "mahal", "murah", "promo"],
         "pelayanan": ["layan", "barista", "kasir", "antri",
                       "pegawai", "mbak"],
         "rasa": ["kopi", "rasa", "manis", "pahit", "espresso"],
         "suasana": ["tempat", "wifi", "acnya", "meja", "colok",
                     "berisik"]}
Yk = np.array([[any(k in t for k in KUNCI[tp]) for tp in TOPIK]
               for t in d.bersih]).astype(int)

tv = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xl, Xu = tv.fit_transform(d.bersih[latih]), tv.transform(
    d.bersih[~latih])
ovr = OneVsRestClassifier(LogisticRegression(C=10, max_iter=2000))
Yp = ovr.fit(Xl, Y[latih]).predict(Xu)
Yu, Yku = Y[~latih], Yk[~latih]
print("F1 per topik (uji):   kata kunci   logistik")
for j, tp in enumerate(TOPIK):
    print(f"  {tp:18s} {f1_score(Yu[:, j], Yku[:, j]):.3f}"
          f"      {f1_score(Yu[:, j], Yp[:, j]):.3f}")
mikro = [f1_score(Yu, y, average="micro") for y in (Yku, Yp)]
tepat = [(Yu == y).all(1).mean() for y in (Yku, Yp)]
print(f"  {'mikro-F1':18s} {mikro[0]:.3f}      {mikro[1]:.3f}")
print(f"  {'semua tepat':18s} {tepat[0]:.3f}      {tepat[1]:.3f}")

neg = (d.sentimen == "negatif").to_numpy()
clf = LogisticRegression(C=5, max_iter=2000, class_weight="balanced")
p_cv = cross_val_predict(clf, Xl, neg[latih], cv=5,
                         method="predict_proba")[:, 1]
ambang = max(a for a in np.arange(.05, .95, .01)
             if (p_cv[neg[latih]] >= a).mean() >= .95)
pu = clf.fit(Xl, neg[latih]).predict_proba(Xu)[:, 1] >= ambang
print(f"\nambang keluhan (recall CV >= 0,95): {ambang:.2f}")
print(f"uji: recall {recall_score(neg[~latih], pu):.3f}, presisi "
      f"{precision_score(neg[~latih], pu):.3f}, ditandai "
      f"{pu.mean():.1%} ulasan")

Xs = tv.transform(d.bersih)
d["keluhan"] = clf.predict_proba(Xs)[:, 1] >= ambang
T = pd.DataFrame(ovr.predict(Xs), columns=TOPIK, index=d.index)
bulan = d.tanggal.dt.to_period("M")
tren = T[d.keluhan].groupby(bulan[d.keluhan]).sum()
tren = tren.div(d.groupby(bulan).size(), axis=0) * 100
print("\nkeluhan per topik per 100 ulasan:")
print(tren.round(1).to_string())
benar = pd.DataFrame(Y, columns=TOPIK)[neg].groupby(bulan[neg]).sum()
benar = benar.div(d.groupby(bulan).size(), axis=0) * 100
r = np.corrcoef(tren.to_numpy().ravel(), benar.to_numpy().ravel())
print(f"korelasi dengan hitungan label manual: {r[0, 1]:.3f}")
n = d.groupby(bulan).size()
p0 = tren.mean() / 100
se = np.sqrt(np.outer(1 / n, p0 * (1 - p0)))
bka = 100 * (p0.to_numpy() + 3 * se)
sinyal = tren > bka
print("di atas batas kendali p:",
      [(str(b), t) for b, t in sinyal.stack()[sinyal.stack()].index]
      or "tidak ada")

baru = ["kurirnya nyasar, kopi dateng telat dan dingin",
        "baristanya cuek bgt, dipanggil diem aja",
        "aplikasinya ngelag pas checkout",
        "harganya bikin dompet nangis",
        "kopinya enak, tempatnya gerah",
        "ga worth it, mending beli sachet"]
Pb = ovr.predict_proba(tv.transform([bersihkan(t) for t in baru]))
print("\nulasan dengan kata-kata baru:")
for t, p in zip(baru, Pb):
    top = np.argsort(p)[::-1][:2]
    tebak = ", ".join(f"{TOPIK[j]} {p[j]:.2f}" for j in top)
    print(f"  {t[:38]:38s} {tebak}")
tren.assign(t=range(1, len(tren) + 1)).to_csv(
    "gambar/bab25/tren.dat", sep=" ", index=False,
    float_format="%.2f")